In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================================
# STEP 1 : Install all required Python libraries
# ==========================================================

# CrewAI Framework
!pip install crewai==0.36.1 -q

# LangChain Framework
!pip install langchain==0.2.16 -q

# Groq Integration with LangChain
!pip install langchain-groq==0.1.9 -q

# Display Markdown in Colab
!pip install ipython -q

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# ==========================================================
# STEP 2 : Import all required libraries
# ==========================================================

# Used to work with environment variables
import os

# Used for asynchronous execution
import asyncio

# CrewAI main classes
from crewai import Agent
from crewai import Task
from crewai import Crew

# Groq LLM
from langchain_groq import ChatGroq

# Display beautiful Markdown output
from IPython.display import Markdown

# Used for securely entering API key
from getpass import getpass

In [ ]:
# ==========================================================
# STEP 3 : Store the Groq API Key securely
# ==========================================================

# Ask the user to enter the API key
groq_key = getpass("Enter your Groq API Key: ")

# Store the key in an environment variable
os.environ["GROQ_API_KEY"] = groq_key

In [ ]:
# ==========================================================
# STEP 4 : Initialize the Groq Large Language Model (LLM)
# ==========================================================

# Create an instance of ChatGroq
# ChatGroq acts as a bridge between LangChain and the Groq API.
llm = ChatGroq(

    # API key stored securely in the environment variable
    groq_api_key=os.environ["GROQ_API_KEY"],

    # Llama 3 model provided by Groq
    model_name="llama-3.3-70b-versatile",

    # Controls randomness.
    # Lower value = More deterministic
    # Higher value = More creative
    temperature=0.5
)

In [ ]:
# ==========================================================
# STEP 5 : Test the Groq LLM
# ==========================================================

# Ask a simple question
response = llm.invoke(
    "Explain Artificial Intelligence in one sentence."
)

# Print the response generated by the model
print(response.content)

In [ ]:
# ==========================================================
# STEP 6 : Create the Planner Agent
# ==========================================================

planner = Agent(

    # The job title of the AI agent
    role="Content Planner",

    # The objective the agent should always try to achieve
    goal="Plan engaging and factually accurate content about {topic}.",

    # Background information that shapes the agent's personality and behavior
    backstory="""
    You are an experienced Content Strategist.

    Your responsibility is to research the given topic,
    identify the latest trends,
    understand the target audience,
    gather reliable information,
    and prepare a detailed content outline.

    Your work will be used by another AI agent
    to write the final blog article.
    """,

    # Prevent this agent from assigning work to other agents
    allow_delegation=False,

    # Show detailed execution logs
    verbose=True,

    # Assign the Groq LLM created earlier
    llm=llm
)

In [ ]:
# ==========================================================
# STEP 7 : Create the Content Writer Agent
# ==========================================================

writer = Agent(

    # The role assigned to this AI agent
    role="Content Writer",

    # Primary objective of the agent
    goal="Write engaging, informative, and factually accurate blog articles about {topic}.",

    # Background that defines the writing style and behavior
    backstory="""
    You are a professional technical content writer.

    You receive a detailed content outline from the Content Planner.

    Your responsibility is to transform that outline into a
    high-quality blog article that is:

    • Easy to understand
    • Well structured
    • SEO friendly
    • Factually correct
    • Interesting to read

    You always maintain a professional tone,
    provide smooth transitions between sections,
    and never invent facts that are not supported by the planner's outline.
    """,

    # Prevent this agent from assigning work to others
    allow_delegation=False,

    # Display execution logs
    verbose=True,

    # Assign the Groq language model
    llm=llm
)

In [ ]:
# ==========================================================
# STEP 8 : Create the Editor Agent
# ==========================================================

editor = Agent(

    # The professional role of this AI
    role="Editor",

    # Main objective
    goal="Review and improve blog articles before publication.",

    # Professional background of the editor
    backstory="""
    You are a senior editor working for a well-known technology magazine.

    Your responsibility is to carefully review articles written by the Content Writer.

    You improve:

    • Grammar
    • Spelling
    • Sentence structure
    • Readability
    • Logical flow
    • Professional tone

    You also verify that the article remains factually consistent,
    avoids exaggerated claims,
    and follows journalistic best practices.

    You never rewrite the entire article unless absolutely necessary.
    Instead, you refine and polish the existing content.
    """,

    # Do not assign work to other agents
    allow_delegation=False,

    # Print execution logs
    verbose=True,

    # Use the Groq LLM
    llm=llm
)

In [ ]:
# ==========================================================
# STEP 9 : Task for the Planner Agent
# ==========================================================

plan = Task(

    # Instructions given to the Planner
    description="""
    Your task is to research the topic: {topic}

    Perform the following steps:

    1. Identify the latest trends.

    2. Identify important technologies.

    3. Identify the target audience.

    4. List important challenges.

    5. Prepare a detailed blog outline.

    6. Suggest SEO keywords.

    7. Mention useful references if available.
    """,

    # Expected output from this task
    expected_output="""
    A detailed content plan including:

    - Introduction

    - Main headings

    - Subheadings

    - Target audience

    - SEO keywords

    - References

    - Blog outline
    """,

    # Which agent should execute this task?
    agent=planner
)

In [ ]:
# ==========================================================
# STEP 10 : Task for the Writer Agent
# ==========================================================

write = Task(

    # Instructions for the Content Writer
    description="""
    Using the content plan created by the Content Planner,
    write a complete blog article on the topic: {topic}

    Your article should include:

    1. An engaging title.

    2. A compelling introduction.

    3. Proper headings and subheadings.

    4. Clear explanations with examples.

    5. Natural use of SEO keywords.

    6. A strong conclusion.

    7. A Call-To-Action (CTA) at the end.

    Ensure the article is professional,
    informative,
    and easy to understand.
    """,

    # What we expect from this task
    expected_output="""
    A complete blog article in Markdown format
    ready for publication.
    """,

    # This task depends on the planner's task
    context=[plan],

    # Assign this task to the Writer Agent
    agent=writer
)

In [ ]:
# ==========================================================
# STEP 11 : Task for the Editor Agent
# ==========================================================

edit = Task(

    # Instructions for the Editor
    description="""
    Carefully review the blog article written by the Content Writer.

    Your responsibilities include:

    1. Correct grammar mistakes.

    2. Improve sentence structure.

    3. Improve readability.

    4. Ensure a professional tone.

    5. Remove repetitive content.

    6. Verify logical flow between sections.

    7. Ensure the article is ready for publication.

    Do not change the meaning of the article.
    Improve the quality while preserving the original intent.
    """,

    # Expected result
    expected_output="""
    A polished blog article in Markdown format,
    free of grammatical errors,
    professionally written,
    and ready for publication.
    """,

    # Use the Writer's output as input
    context=[write],

    # Assign this task to the Editor Agent
    agent=editor
)

In [ ]:
# ==========================================================
# STEP 12 : Create the Crew
# ==========================================================

from crewai import Process

# Create the Crew
crew = Crew(

    # List of all AI agents
    agents=[
        planner,
        writer,
        editor
    ],

    # List of tasks to execute
    tasks=[
        plan,
        write,
        edit
    ],

    # Tasks will execute one after another
    process=Process.sequential,

    # Display detailed execution logs
    verbose=2
)

In [ ]:
# ==========================================================
# STEP 13 : Execute the Crew
# ==========================================================

# Topic that all agents will work on
topic = input("Enter the blog topic: ")

# Start the Crew
result = crew.kickoff(
    inputs={
        "topic": topic
    }
)

# Print the final output
print(result)

In [ ]:
# ==========================================================
# STEP 14 : Display Beautiful Markdown
# ==========================================================

from IPython.display import Markdown

Markdown(str(result))